# ZiguratIP on Google Colab

Build and run **ZiguratIP** — a single-process database (*Zigurat*), language (*Parsi*),
and web server (*Zeytun*) written in dependency-free C++17 — inside a Colab VM.

Two ways to look at it: **inline over loopback**, which is what a run does by default and
exposes nothing, or through a **Cloudflare tunnel**, which gives you a real URL and is
opt-in (`TUNNEL = True` in section 6).

- **Zigurat** — MVCC storage engine, binary protocol on port **2160**
- **Parsi** — SQL-like language compiled to a `.so` the server `dlopen`s
- **Zeytun** — HTTP server for static files and `.zt` pages on port **2190**

Read [`TUTORIAL.md`](TUTORIAL.md) beside this notebook for what each step is doing and
how to go further.

## Why a tunnel, and not Colab's port proxy

Colab has a built-in proxy — `google.colab.kernel.proxyPort(2190)` — and it does not
work for this. Four rounds of "blank page" were spent on it, and the diagnostic that
settled it was simple: mark the size of `server.log`, open the proxy URL in a browser,
and look again. **Nothing was ever appended.** The request never reached Zeytun at all,
so no change to ZiguratIP could have fixed it. Those diagnostic cells have been removed;
this notebook uses the tunnel instead.

A tunnel goes the other way. `cloudflared` opens an *outbound* connection to Cloudflare
and they hand back a public hostname that forwards down it — nothing has to reach *in*,
so Colab's routing stops mattering. Because the whole host maps to Zeytun, the pages'
relative links and images (`src="zeytun.png"`, `href="setup.zt"`) resolve properly, which
they do not through a path-prefixed proxy.

## 🔓 What you are putting on the internet

A quick tunnel has **no account, no authentication, and no access control**. Anyone with
the URL reaches the server, and the URL travels in the clear through Cloudflare's
infrastructure — it is unguessable, but it is not a secret.

The server behind it is a sandbox: no authentication in front of any page, no write-ahead
log, and a demo dataset. **Do not point one at anything
you care about, and stop it when you are done.**

None of this happens unless you ask for it: `TUNNEL` is `False` and section 7 shows the
same pages without exposing anything.

When you do ask, cell 6 still refuses if either of the two things that would turn this from
"exposed sandbox" into "arbitrary code execution" is present. Read that cell before
overriding it.

## 1 · Build

> **Any branch builds now.** `BRANCH` below defaults to `master`, which is where the Linux
> build fixes live since the `colab` branch was merged into it: fixed-width integer types
> missing `<cstdint>`, `htonl` and friends coming back as glibc macros, and two libraries
> that never declared what they link against — none of which macOS showed. Worse, every
> recipe in the top-level `Makefile` was prefixed with `@-`, so failures were stepped over
> and the run still ended with `******* all done *******`; a clean checkout produced 2 of
> 14 libraries and no executables while reporting success. Point `BRANCH` at anything older
> than that merge and it will appear to build and then have nothing to run.

Colab already ships `g++` and `make`; the `apt-get` line is just insurance.

In [ ]:
import os, subprocess

REPO   = 'https://github.com/saman-pasha/ZiguratIP.git'
BRANCH = 'master'  # the Linux build fixes are merged; older branches do not build
SRC    = '/content/ZiguratIP'

!apt-get -qq install -y build-essential >/dev/null 2>&1 || true

# Check the branch exists before cloning. A clone of a branch that is not
# there fails quietly enough that every later cell blames something else.
ls = subprocess.run(['git','ls-remote','--heads',REPO,BRANCH],
                    capture_output=True, text=True)
assert BRANCH in ls.stdout, f'branch {BRANCH!r} not found on the remote:\n{ls.stdout}{ls.stderr}'
print(f'branch {BRANCH} found')

# Clone, or bring an existing clone up to date. Re-running this notebook
# in a session that already cloned used to skip straight past here and
# rebuild whatever was fetched the first time, so a fix pushed since then
# never arrived and the symptom it fixed was still there.
if not os.path.isdir(SRC):
    !git clone --depth 1 -b $BRANCH $REPO $SRC
else:
    !git -C $SRC fetch --depth 1 origin $BRANCH
    !git -C $SRC checkout -B $BRANCH FETCH_HEAD
assert os.path.isfile(f'{SRC}/Makefile'), 'clone produced no tree'

head = subprocess.run(['git','-C',SRC,'log','-1','--format=%h %s'],
                      capture_output=True, text=True).stdout.strip()
print('building:', head)

# A rebuild has to see the new sources, and the object files from the
# previous run are older than nothing the Makefile knows to check.
!make -C $SRC clean >/dev/null 2>&1 || true

%cd /content/ZiguratIP
!make MODE=Release 2>&1 | tail -n 5

# make exits 0 even when projects fail -- every recipe in the top-level
# Makefile is prefixed with @-, so failures are stepped over and it still
# prints 'all done'. Check what was actually produced instead.
libs = sorted(f for f in os.listdir('home/lib') if f.endswith('.so'))
bins = sorted(os.listdir('home/bin'))
print(f'\nlibraries: {len(libs)} (expect 14)')
print(f'executables: {bins}')
missing = {'Test','ca','parsi','ziguratip'} - set(bins)
assert not missing and len(libs) >= 14, f'BUILD INCOMPLETE -- missing {missing or "libraries"}'
print('build OK')

## 2 · Runtime environment

`ZIGURATIP_HOME` is both the install prefix and the runtime home. On **Linux** the shared
libraries are found via `LD_LIBRARY_PATH` (not macOS's `DYLD_LIBRARY_PATH`). A C++ compiler
must also stay on `PATH` — Parsi pages are compiled to a `.so` *at request time*, not only
at build time.

In [ ]:
os.environ['ZIGURATIP_HOME']  = '/content/ZiguratIP/home'
os.environ['LD_LIBRARY_PATH'] = os.environ['ZIGURATIP_HOME'] + '/lib'
assert subprocess.call(['which', 'c++']) == 0, 'a C++ compiler must be on PATH for runtime Parsi compilation'
print('ZIGURATIP_HOME =', os.environ['ZIGURATIP_HOME'])
print('binaries:')
!ls -1 $ZIGURATIP_HOME/bin

## 3 · (Optional) restore from Google Drive

Colab VMs are ephemeral, so to keep a database between sessions set
`PERSIST = True` and a snapshot is copied from Drive onto local disk **before**
the server starts. The engine never runs against the Drive mount.

**Three directories, not one.** `home/data` alone is useless, and this is worth
being concrete about because it used to save only that: the rows come back, the
server starts happily, and every page answers **404**. Nothing can read them.

| directory | what it is | without it |
|---|---|---|
| `home/data` | the page store — the rows | there is no database |
| `home/catalog` | one `.conf` per object: its name, hash key, what it REQUIREs | the compiler cannot resolve `REQUIRES`, so nothing new links |
| `home/ld` | the compiled `.so` the server `dlopen`s by name | every page and procedure is a 404 |

They are one unit: a store, the metadata describing it, and the code that reads
it. Saving a third of it saves nothing usable.

**A snapshot is tied to the build that made it.** A compiled object links against
`libCore.so`, `libHTTP.so`, `libMVCCS.so` and `libType.so` by bare name, with no
version — so objects built from one commit and loaded by libraries built from
another can fail at `dlopen` with an undefined symbol. The snapshot records the
commit it came from and this cell says so when they differ; recompiling with
`demo/build.sh` is the fix.

In [ ]:
PERSIST = False  # set True to keep the database across sessions via Drive
DRIVE_BACKUP = '/content/drive/MyDrive/ziguratip-backup'

# The three that travel together. See the table above for why.
SNAPSHOT_DIRS = ['data', 'catalog', 'ld']


def current_commit():
    """The commit this checkout is built from, or '' if it cannot be told."""
    done = subprocess.run(['git', '-C', SRC, 'rev-parse', 'HEAD'],
                          capture_output=True, text=True)
    return done.stdout.strip() if done.returncode == 0 else ''


if PERSIST:
    from google.colab import drive
    drive.mount('/content/drive')

    if not os.path.isdir(DRIVE_BACKUP):
        print('no snapshot in ' + DRIVE_BACKUP +
              ' yet -- a fresh store will be created on first use.')
    else:
        # The manifest is what makes a mismatch reportable rather than a crash
        # twenty minutes later. Written by the snapshot cell.
        manifest = os.path.join(DRIVE_BACKUP, 'MANIFEST')
        saved_commit = ''
        if os.path.exists(manifest):
            for line in open(manifest):
                if line.startswith('commit:'):
                    saved_commit = line.split(':', 1)[1].strip()

        restored = []
        for name in SNAPSHOT_DIRS:
            source = os.path.join(DRIVE_BACKUP, name)
            if not os.path.isdir(source):
                continue
            target = os.path.join(os.environ['ZIGURATIP_HOME'], name)
            subprocess.run(['rm', '-rf', target], check=True)
            subprocess.run(['cp', '-a', source, target], check=True)
            restored.append(name)

        if not restored:
            print(DRIVE_BACKUP + ' exists but holds none of ' +
                  ', '.join(SNAPSHOT_DIRS) + ' -- nothing restored.')
        else:
            print('restored:', ', '.join(restored))
            missing = [d for d in SNAPSHOT_DIRS if d not in restored]
            if missing:
                # Naming it now beats a 404 later that looks like a bad URL.
                print()
                print('MISSING FROM THE SNAPSHOT:', ', '.join(missing))
                if 'ld' in missing:
                    print('  Without ld/ every page and procedure answers 404.')
                if 'catalog' in missing:
                    print('  Without catalog/ nothing new will link against what is there.')
                print('  Run demo/build.sh (cell 4) to rebuild them.')

            here = current_commit()
            if saved_commit and here and saved_commit != here:
                print()
                print('SNAPSHOT IS FROM A DIFFERENT BUILD')
                print('  snapshot:', saved_commit[:12])
                print('  this tree:', here[:12])
                print('  Compiled objects link against libCore/libHTTP/libMVCCS/libType')
                print('  by bare name, so they may fail to load with an undefined symbol.')
                print('  If a page does, recompile it -- demo/build.sh in cell 4.')
            elif not saved_commit:
                print()
                print('(no MANIFEST in the snapshot -- it predates this check, so which')
                print(' build it came from is unknown.)')
else:
    print('PERSIST is off -- a fresh store is created on first use.')


## 4 · Build the demo objects

`demo/build.sh` compiles the demo tables and `.zt` pages with the **offline** `parsi` compiler.
It starts nothing — it just produces the shared objects the server will load.

Note this is the supported way to compile. Compiling *over the network* is refused by default
(`COMPILER/REMOTE_MODE`), because it runs a C++ compiler and linker on whatever a client sends,
which is arbitrary code execution as the server's user. Leave it off.

In [ ]:
!chmod +x demo/build.sh Test/*.sh
!./demo/build.sh 2>&1 | tail -n 12

## 4b · (Optional) accept compiles over HTTP

**Skip this unless you are doing the tunnel milestone in
[`TUTORIAL.md`](TUTORIAL.md).** It is what lets you edit Parsi on your own
machine and compile it onto this VM — and it is the one arrangement in this
notebook that hands the compiler to whoever has the URL.

It needs two things the rest of the notebook deliberately leaves off:

1. **`COMPILER/REMOTE_MODE = TRUE`** in `ziguratip.conf`.
2. **`System/`'s objects compiled**, including `compiler.parsi` — the page that
   POSTs `code` to the compiler.

Together they mean: anyone who reaches `<tunnel>/compiler.zt` can compile and
run C++ inside this VM, as the user running the server. There is no
authentication in front of it, and the tunnel URL is not a secret.

That is not a side effect to be worked around — it *is* the feature. A Colab VM
has no inbound port and a quick tunnel carries only HTTP, so this is the only way
to compile onto it from outside. Do it on a throwaway VM and stop it when you are
done.

`ENABLE_REMOTE_COMPILE = False` leaves everything as it was.

In [ ]:
ENABLE_REMOTE_COMPILE = False   # read the cell above before setting this

import os, re, subprocess

CONF = os.path.join(os.environ['ZIGURATIP_HOME'], 'etc', 'ziguratip.conf')

if not ENABLE_REMOTE_COMPILE:
    print('left off. The server compiles nothing that arrives over the network.')
else:
    # REMOTE_MODE, rewritten in place. Matching the key rather than a whole line
    # so an edited or reformatted conf still works, and only outside comments so
    # the explanation above the setting is not what gets changed.
    text = open(CONF).read()
    new, n = re.subn(r'(?m)^(\s*REMOTE_MODE\s*:\s*)FALSE\b', r'\1TRUE', text)
    if n:
        open(CONF, 'w').write(new)
        print('COMPILER/REMOTE_MODE -> TRUE in', CONF)
    elif re.search(r'(?m)^\s*REMOTE_MODE\s*:\s*TRUE\b', text):
        print('COMPILER/REMOTE_MODE is already TRUE')
    else:
        raise SystemExit('could not find COMPILER/REMOTE_MODE in ' + CONF)

    # The System objects, in dependency order -- each REQUIREs the ones before.
    order = ['htmldrawer', 'connector', 'compilerdrawer', 'compiler']
    for name in order:
        source = 'System/%s.parsi' % name
        done = subprocess.run([os.environ['ZIGURATIP_HOME'] + '/bin/parsi', source],
                              capture_output=True, text=True, env=os.environ)
        if done.returncode != 0:
            print('FAILED on', source)
            print(done.stderr.strip() or done.stdout.strip()[-800:])
            break
        print('compiled', source)
    else:
        print()
        print('RESTART THE SERVER (cell 5) so it reads the new setting, then open')
        print('the tunnel. <tunnel-url>/compiler.zt is now a compiler anyone can reach.')


## 5 · Start the server (background)

The server blocks and listens forever, so it can't own a notebook cell — we launch it as a
background process and tail its log. It prints what it loaded, then listens on **2160**
(binary) and **2190** (HTTP).

In [ ]:
import time

# stop a previous instance if this cell is re-run
try:
    srv.terminate(); srv.wait(timeout=5)
except Exception:
    pass

srv = subprocess.Popen(['./home/bin/ziguratip'],
                       stdout=open('server.log', 'w'),
                       stderr=subprocess.STDOUT,
                       env=os.environ)
time.sleep(2.5)
print(open('server.log').read())
assert srv.poll() is None, 'server exited — see the log above'

## 5b · Check the server answers, before handing out a URL

A tunnel forwards to port 2190 whether or not anything is listening there, so a dead
server and a healthy one look identical from a browser — both render a blank page. This
asks over loopback first and says plainly which of the two it is.

In [ ]:
import urllib.request, time

code_, body = None, b''
for attempt in range(10):
    try:
        with urllib.request.urlopen('http://127.0.0.1:2190/', timeout=5) as r:
            code_, body = r.status, r.read()
        break
    except Exception as e:
        err = e
        time.sleep(1)

if code_ is None:
    print('SERVER IS NOT ANSWERING on 127.0.0.1:2190 --', err)
    print()
    print('Nothing below will work until this does. The log:')
    print(open('server.log').read()[-2000:])
else:
    print('server answered: HTTP %s, %d bytes' % (code_, len(body)))


## 6 · A public URL, through Cloudflare

Set `TUNNEL = True` and run this: `cloudflared` is downloaded, run against
`http://localhost:2190`, and the hostname it is assigned is printed. That URL is how you
reach the demo from a browser.

**It is off by default, and that is deliberate.** Running a notebook top to bottom should
not put a server on the public internet as a side effect. Section 7 renders the same pages
inline over loopback and exposes nothing — reach for the tunnel when you actually want a
URL to open in a browser or hand to somebody.

**The two checks below are not ceremony.** ZiguratIP compiles Parsi to C++ and links it
at request time, so a compiler reachable from the network is arbitrary code execution as
the server's user:

1. **`COMPILER/REMOTE_MODE`** — when `TRUE`, any client that can open the binary port may
   send source to be compiled. It is `FALSE` by default and must stay that way here.
2. **A compiled `Compiler` page** — `System/compiler.parsi` is a *web page* that hands
   `request.post('code')` straight to the compiler. `demo/build.sh` does not build it, so
   a normal run has none; if you have compiled it yourself, it is sitting in `home/ld`
   waiting for anyone with the tunnel URL.

Either one found, and the tunnel does not start. Set `I_UNDERSTAND = True` only if you
have read what that means and the tunnel is not actually reachable by anyone else.

In [ ]:
import os, re, subprocess, time, glob

TUNNEL        = False   # set True to put this server on the public internet
I_UNDERSTAND  = False   # override the safety checks below (do not)

CF  = '/content/cloudflared'
LOG = '/content/cloudflared.log'
HOME = os.environ.get('ZIGURATIP_HOME', '/content/ZiguratIP/home')


def unsafe_to_expose():
    """What must not be reachable from the internet. Answers a list of reasons."""
    reasons = []

    conf = os.path.join(HOME, 'etc', 'ziguratip.conf')
    try:
        for line in open(conf):
            bare = line.split('#', 1)[0]
            if 'REMOTE_MODE' in bare and 'TRUE' in bare.upper():
                reasons.append(
                    'COMPILER/REMOTE_MODE is TRUE in %s -- anyone who can open the '
                    'binary port could compile arbitrary C++ as this user' % conf)
                break
    except FileNotFoundError:
        pass

    # PAGE Compiler compiles into lib_COMPILER_.so. The match is LOOSE on purpose --
    # any page reaching the compiler is the same hazard under any name -- so it is
    # worded as something to look at rather than as a claim about what the file does.
    # Run against this repository's own checkout it found two: the compiler page, and
    # the drawer that only exists to render it. Neither belongs behind a tunnel.
    for so in sorted(glob.glob(os.path.join(HOME, 'ld', 'lib_*COMPILER*_.so'))):
        if os.path.basename(so) == 'lib_COMPILER_.so':
            reasons.append('%s is loadable -- System/compiler.parsi is a web page that '
                           'hands request.post(\'code\') to the compiler' % so)
        else:
            reasons.append('%s is loadable and its name says compiler -- check what it '
                           'serves before exposing it' % so)

    return reasons


if not TUNNEL:
    print('TUNNEL is off. Nothing is exposed; use visit() below instead.')
else:
    problems = unsafe_to_expose()
    if problems and not I_UNDERSTAND:
        print('REFUSING TO OPEN A TUNNEL. Found:')
        for p in problems:
            print('  -', p)
        print()
        print('Fix it, or set I_UNDERSTAND = True if you are certain.')
    else:
        if problems:
            print('WARNING -- opening anyway because I_UNDERSTAND is set:')
            for p in problems:
                print('  -', p)
            print()

        if not os.path.exists(CF):
            print('fetching cloudflared ...')
            subprocess.run(['curl', '-sSL', '-o', CF,
                            'https://github.com/cloudflare/cloudflared/releases/latest/'
                            'download/cloudflared-linux-amd64'], check=True)
            os.chmod(CF, 0o755)

        # stop a previous tunnel if this cell is re-run, so a second one does not
        # quietly take a different hostname while the first still holds the port
        old = globals().get('_tunnel')
        if old is not None and old.poll() is None:
            old.terminate()
            try:
                old.wait(timeout=5)
            except Exception:
                old.kill()

        with open(LOG, 'w') as log:
            tunnel = subprocess.Popen(
                [CF, 'tunnel', '--url', 'http://localhost:2190', '--no-autoupdate'],
                stdout=log, stderr=subprocess.STDOUT)
        globals()['_tunnel'] = tunnel

        # cloudflared prints the hostname in a banner once Cloudflare assigns it
        url = None
        for _ in range(45):
            if tunnel.poll() is not None:
                break
            try:
                found = re.findall(r'https://[a-z0-9][a-z0-9-]*\.trycloudflare\.com',
                                   open(LOG).read())
                if found:
                    url = found[0]
                    break
            except FileNotFoundError:
                pass
            time.sleep(2)

        if url:
            print('Open this in a browser:')
            print(' ', url)
            print()
            print('  seed the demo :', url + '/setup.zt')
            print('  then browse   :', url + '/catalog.zt')
            print()
            print('Stop it with:  _tunnel.terminate()')
        else:
            # Say why rather than leaving a dead cell. The usual causes are an egress
            # policy blocking api.trycloudflare.com, or nothing listening on 2190.
            print('NO TUNNEL URL. cloudflared said:')
            try:
                for line in open(LOG).read().splitlines()[-12:]:
                    print('  ', line[:160])
            except FileNotFoundError:
                print('   (no log written at all)')


## 7 · Or stay on loopback, with no tunnel at all

`visit()` fetches a page **and every asset it references** over loopback, embeds them as
`data:` URIs, and renders the result inline. Nothing is exposed, and what you see is what
Zeytun actually served — images included.

This is the safest way to look at the demo, and it works whether or not the tunnel is up.

In [ ]:
import urllib.request, base64, re, mimetypes
from IPython.display import HTML, display

BASE = 'http://127.0.0.1:2190'

def fetch(path):
    with urllib.request.urlopen(BASE + '/' + path.lstrip('/'), timeout=15) as r:
        return r.status, r.read(), r.headers.get('Content-Type', '')

def visit(path='/', quiet=False):
    """Fetch a page, embed everything it references, and show it.

    Nothing here goes through a proxy, so relative references resolve
    against this server rather than against Colab's own host -- which is
    what makes the banner load and the links read correctly.
    """
    status, body, ctype = fetch(path)
    html = body.decode('utf-8', 'replace')
    inlined = 0
    refs = sorted(set(re.findall(
        r'(?:src|href)="(?!https?:|//|#|mailto:|data:)([^"]+)"', html)))
    for ref in refs:
        if ref.endswith('.zt') or ref == '/':
            continue                     # a page: visit() it rather than embed it
        try:
            s, data, ct = fetch(ref)
            if s != 200: continue
            ct = ct or mimetypes.guess_type(ref)[0] or 'application/octet-stream'
            html = html.replace('"%s"' % ref,
                                '"data:%s;base64,%s"' % (ct, base64.b64encode(data).decode()))
            inlined += 1
        except Exception:
            pass
    if not quiet:
        print(f'{path}: HTTP {status}, {len(body)} bytes, {inlined} asset(s) embedded')
    display(HTML(html))
    return status

visit('/')

### Walking the demo

`visit()` takes any path, so the demo can be followed here without a browser:

```python
visit('/setup.zt')     # seed the catalogue -- run once
visit('/catalog.zt')   # the books and authors it created
visit('/lookup.zt')    # queries served from single-column indexes
visit('/report.zt')    # a two-column index
```

Running `setup.zt` a second time reports `unique key 'IDX_DEMO_AUTHORS_NAME'` —
that is the index refusing a duplicate author, which is the point of it.

In [ ]:
visit('/catalog.zt')

## 8 · (Optional) snapshot back to Drive

Only meaningful with `PERSIST = True`, and only **after a clean stop** — run the
next cell first. There is no write-ahead log, so a snapshot taken while the
server is mid-write can be inconsistent.

Saves the same three directories the restore expects, plus a `MANIFEST` naming
the commit they were built from, so a later restore against a different build can
say so instead of failing at `dlopen`.

In [ ]:
# globals().get, because `srv' only exists once cell 5 has run. Naming it
# directly meant a NameError for anyone who opened the notebook to take a
# snapshot of a restored store without starting anything.
_srv = globals().get('srv')

if not PERSIST:
    print('PERSIST is off -- nothing to save.')
elif _srv is not None and _srv.poll() is None:
    # Refusing beats writing a torn store over the good one: there is no WAL,
    # so a snapshot taken mid-write can be inconsistent and this is the copy you
    # would be restoring from next time.
    print('THE SERVER IS STILL RUNNING. Stop it first (cell 9), then run this.')
else:
    os.makedirs(DRIVE_BACKUP, exist_ok=True)

    saved = []
    for name in SNAPSHOT_DIRS:
        source = os.path.join(os.environ['ZIGURATIP_HOME'], name)
        if not os.path.isdir(source):
            print('skipping %s -- not there' % name)
            continue
        target = os.path.join(DRIVE_BACKUP, name)
        subprocess.run(['rm', '-rf', target], check=True)
        subprocess.run(['cp', '-a', source, target], check=True)
        saved.append(name)

    with open(os.path.join(DRIVE_BACKUP, 'MANIFEST'), 'w') as f:
        f.write('commit: %s\n' % current_commit())
        f.write('dirs: %s\n' % ' '.join(saved))

    print('snapshot written to', DRIVE_BACKUP)
    print('  ', ', '.join(saved))
    for name in saved:
        n = sum(len(files) for _, _, files in os.walk(os.path.join(DRIVE_BACKUP, name)))
        print('   %-8s %d files' % (name, n))


## 9 · Stop the server

In [ ]:
try:
    srv.terminate(); srv.wait(timeout=5)
    print('server stopped.')
except Exception as e:
    print('nothing running:', e)

## 10 · Stop the tunnel

The tunnel outlives the server: stopping ZiguratIP leaves `cloudflared` up and the URL
answering 502, which looks like a broken server rather than a stopped one. Stop both.

In [ ]:
t = globals().get('_tunnel')
if t is not None and t.poll() is None:
    t.terminate()
    try:
        t.wait(timeout=5)
    except Exception:
        t.kill()
    print('tunnel stopped -- the URL is dead now.')
else:
    print('no tunnel running.')


## Notes & limits

- **Durability:** the storage engine has no WAL/fsync, so a hard runtime kill mid-commit can
  corrupt the store. Snapshot to Drive only after a clean stop.
- **Concurrency:** thread-per-connection on a pool of 64, backlog 64. It was 5, which one
  visitor could saturate on their own — six parallel connections is what a browser opens for
  a single page, and the seventh waited on the first six.
- **TLS:** the server speaks interoperable TLS now — ECDHE and AEAD only, static RSA refused
  outright, TLS 1.3 negotiated, and a browser needs no certificate of its own if the port is
  set `TLS_CLIENT_AUTH: NONE`. Leave `HTTP/TLS_MODE: FALSE` **in Colab** anyway: the proxy
  speaks plain HTTP to your port and terminates TLS itself, so turning it on here just means
  two layers talking past each other. On a real host, turn it on.
- **Reaching it from a browser in Colab:** you cannot, and it is not this server's doing. The
  diagnostic above confirmed the request never arrives — Colab's edge answers `404 page not
  found`, which is the stock body Go's net/http writes, before anything gets here. `visit()`
  is the way to use the demo from a Colab session.
- **Drive:** never set `HOME_PATH`/data onto the `/content/drive` mount — FUSE breaks the
  pager's random in-place writes and `dlopen`. Local disk for running, Drive for snapshots.
- **Security:** see `doc/outstanding.md`. The short version: no authentication in front of
  any page, no write-ahead log, and vendored zlib 1.2.11. `ECHO` and `SELECT` do escape the
  values they are handed — a column holding `<script>` arrives as text, see `doc/page.md` —
  so the markup hole you have to open yourself, with `` `Zigurat::`Utility::`raw() ``.
  Treat every run as disposable.